# 유휴 공유재산 중심 EDA

## 분석 목적

부산 유휴 공유재산의 면적·지목·공간분포를 파악하고,
지역수요 프로필과 결합하여 활용유형 설계에 필요한
재산 및 입지 특성을 탐색한다.

단순히 유휴재산이 많은 지역을 찾는 것이 아니라,

- 어떤 형태의 유휴재산이 존재하는가?
- 지역별 재산 특성은 어떻게 다른가?
- 지역의 생활수요와 재산 조건 사이에 어떤 조합이 존재하는가?

를 확인한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_PATH = "/content/drive/MyDrive/부산_BigData_공모전/"

idle = pd.read_csv(
    BASE_PATH + "02_processed_data/유휴공유재산_지역수요결합.csv",
    encoding="utf-8-sig"
)

print("shape:", idle.shape)
print("\ncolumns:")
print(idle.columns.tolist())

print("\n[핵심 결측]")
check_cols = [
    "gugun",
    "lctnAddr",
    "zoneArea",
    "land_type",
    "h_code",
    "행정동명"
]

print(idle[check_cols].isna().sum())

print("\n[위치 매칭 상태]")
print(idle["geocode_status"].value_counts(dropna=False))

print("\n[구군별 건수]")
print(idle["gugun"].value_counts())

display(idle.head())

In [ ]:
# 1. 면적 분포
print("[면적 기술통계]")
display(idle["zoneArea"].describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
))

print("\n면적 0 이하:", (idle["zoneArea"] <= 0).sum())

print("\n[면적 구간별 건수]")
area_bins = [-np.inf, 10, 30, 50, 100, 300, 1000, np.inf]
area_labels = [
    "10㎡ 이하",
    "10~30㎡",
    "30~50㎡",
    "50~100㎡",
    "100~300㎡",
    "300~1000㎡",
    "1000㎡ 초과"
]

idle["면적구간"] = pd.cut(
    idle["zoneArea"],
    bins=area_bins,
    labels=area_labels
)

print(idle["면적구간"].value_counts().sort_index())


# 2. 지목 분포
print("\n[지목 상위 20]")
print(
    idle["land_type"]
    .fillna("미제공")
    .value_counts()
    .head(20)
)


# 3. 구군별 유휴재산 수와 면적
district_summary = (
    idle.groupby("gugun")
    .agg(
        유휴재산수=("lctnAddr", "size"),
        면적중앙값=("zoneArea", "median"),
        면적평균=("zoneArea", "mean"),
        총면적=("zoneArea", "sum"),
        지목결측=("land_type", lambda x: x.isna().sum())
    )
    .sort_values("유휴재산수", ascending=False)
)

display(district_summary)


# 4. 구군 × 주요 지목
land_district = pd.crosstab(
    idle["gugun"],
    idle["land_type"].fillna("미제공")
)

display(land_district)


# 5. 행정동별 유휴재산 분포
dong_summary = (
    idle.dropna(subset=["행정동명"])
    .groupby("행정동명")
    .agg(
        유휴재산수=("lctnAddr", "size"),
        면적중앙값=("zoneArea", "median"),
        총면적=("zoneArea", "sum")
    )
    .sort_values("유휴재산수", ascending=False)
)

print("\n[유휴재산 수 상위 행정동]")
display(dong_summary.head(20))

In [ ]:
# 지목 원본을 보존한 뒤 출처별 코드 표기만 통일
idle["land_type_original"] = idle["land_type"]

print("[정리 전 지목]")
print(
    idle["land_type"]
    .fillna("미제공")
    .value_counts()
)

# 숫자코드-지목 형태만 안전하게 제거
idle["land_type_clean"] = (
    idle["land_type"]
    .astype("string")
    .str.replace(r"^\d+-", "", regex=True)
)

print("\n[정리 후 지목]")
print(
    idle["land_type_clean"]
    .fillna("미제공")
    .value_counts()
)

print("\n[변경된 값]")
changed = idle[
    idle["land_type"].astype("string")
    != idle["land_type_clean"].astype("string")
][["land_type", "land_type_clean"]].drop_duplicates()

display(changed)

In [ ]:
# 면적 0㎡ 자료 확인
zero_area = idle[idle["zoneArea"] <= 0][
    [
        "gugun",
        "lctnAddr",
        "zoneArea",
        "land_type_original",
        "data_source",
        "referenceDate"
    ]
]

display(zero_area)

In [ ]:
# 행정동별 유휴재산 특성 구축
dong_property = (
    idle.dropna(subset=["행정동코드"])
    .groupby(["행정동코드", "행정동명"])
    .agg(
        유휴재산수=("lctnAddr", "size"),
        총면적=("zoneArea", "sum"),
        면적중앙값=("zoneArea", "median"),
        면적평균=("zoneArea", "mean"),
        최소면적=("zoneArea", "min"),
        최대면적=("zoneArea", "max"),
        대_필지수=("land_type_clean", lambda x: (x == "대").sum()),
        도로_필지수=("land_type_clean", lambda x: (x == "도로").sum())
    )
    .reset_index()
)

# 행정동별 지역수요는 중복 제거
demand_cols = [
    "행정동코드",
    "전체활동인구",
    "20대미만_활동인구",
    "20대미만_구성비",
    "60대이상_활동인구",
    "60대이상_구성비",
    "아동돌봄시설_전체",
    "고령층생활서비스_전체",
    "주거인구_구성비",
    "직장인구_구성비",
    "방문인구_구성비"
]

dong_demand = (
    idle.dropna(subset=["행정동코드"])
    [demand_cols]
    .drop_duplicates(subset=["행정동코드"])
)

dong_analysis = dong_property.merge(
    dong_demand,
    on="행정동코드",
    how="left",
    validate="one_to_one"
)

print("유휴재산 존재 행정동:", len(dong_analysis))
print("중복 코드:", dong_analysis["행정동코드"].duplicated().sum())

display(dong_analysis.head())

In [ ]:
# 행정동의 대표 활동특성 분류
activity_cols = {
    "주거형": "주거인구_구성비",
    "직장형": "직장인구_구성비",
    "방문형": "방문인구_구성비"
}

dong_analysis["대표활동유형"] = (
    dong_analysis[
        list(activity_cols.values())
    ]
    .idxmax(axis=1)
    .map({v: k for k, v in activity_cols.items()})
)

print("[대표 활동유형]")
print(dong_analysis["대표활동유형"].value_counts())

activity_property_summary = (
    dong_analysis
    .groupby("대표활동유형")
    .agg(
        행정동수=("행정동코드", "size"),
        유휴재산수_중앙값=("유휴재산수", "median"),
        필지면적_중앙값=("면적중앙값", "median"),
        행정동총면적_중앙값=("총면적", "median"),
        대_필지수_중앙값=("대_필지수", "median"),
        도로_필지수_중앙값=("도로_필지수", "median")
    )
)

display(activity_property_summary)

In [ ]:
# 행정동 단위 지역수요 × 유휴재산 특성 탐색
corr_cols = [
    "유휴재산수",
    "총면적",
    "면적중앙값",
    "대_필지수",
    "도로_필지수",
    "전체활동인구",
    "20대미만_활동인구",
    "20대미만_구성비",
    "60대이상_활동인구",
    "60대이상_구성비",
    "아동돌봄시설_전체",
    "고령층생활서비스_전체",
    "주거인구_구성비",
    "직장인구_구성비",
    "방문인구_구성비"
]

corr = (
    dong_analysis[corr_cols]
    .corr(method="spearman")
)

demand_vars = [
    "전체활동인구",
    "20대미만_활동인구",
    "20대미만_구성비",
    "60대이상_활동인구",
    "60대이상_구성비",
    "주거인구_구성비",
    "직장인구_구성비",
    "방문인구_구성비"
]

property_vars = [
    "유휴재산수",
    "총면적",
    "면적중앙값",
    "대_필지수",
    "도로_필지수"
]

display(
    corr.loc[demand_vars, property_vars]
    .round(3)
)

In [ ]:
# 면적 0㎡ 자료는 삭제하지 않고 유효 여부만 표시
idle["area_valid"] = idle["zoneArea"] > 0

print(idle["area_valid"].value_counts())

In [ ]:
# 부산 전체 생활인구를 기준으로 활동유형별 기준 구성비 계산
profile_unique = (
    idle.dropna(subset=["행정동코드"])
    [
        [
            "행정동코드",
            "평균주거인구수",
            "평균직장인구수",
            "평균방문인구수",
            "전체활동인구"
        ]
    ]
    .drop_duplicates(subset=["행정동코드"])
)

city_residential_share = (
    profile_unique["평균주거인구수"].sum()
    / profile_unique["전체활동인구"].sum()
)

city_workplace_share = (
    profile_unique["평균직장인구수"].sum()
    / profile_unique["전체활동인구"].sum()
)

city_visit_share = (
    profile_unique["평균방문인구수"].sum()
    / profile_unique["전체활동인구"].sum()
)

print("부산 전체 기준")
print("주거:", round(city_residential_share, 4))
print("직장:", round(city_workplace_share, 4))
print("방문:", round(city_visit_share, 4))

In [ ]:
# 부산 206개 전체 행정동 프로필 사용
profile_all = pd.read_csv(
    BASE_PATH + "02_processed_data/지역수요프로필_행정동별.csv",
    encoding="utf-8-sig"
)

city_residential_share = (
    profile_all["평균주거인구수"].sum()
    / profile_all["전체활동인구"].sum()
)

city_workplace_share = (
    profile_all["평균직장인구수"].sum()
    / profile_all["전체활동인구"].sum()
)

city_visit_share = (
    profile_all["평균방문인구수"].sum()
    / profile_all["전체활동인구"].sum()
)

print("[부산 206개 행정동 전체 기준]")
print("주거 비중:", round(city_residential_share, 4))
print("직장 비중:", round(city_workplace_share, 4))
print("방문 비중:", round(city_visit_share, 4))

In [ ]:
# 부산 평균 대비 상대적 활동특성
dong_analysis["주거_LQ"] = (
    dong_analysis["주거인구_구성비"] / city_residential_share
)

dong_analysis["직장_LQ"] = (
    dong_analysis["직장인구_구성비"] / city_workplace_share
)

dong_analysis["방문_LQ"] = (
    dong_analysis["방문인구_구성비"] / city_visit_share
)

lq_cols = {
    "주거특화": "주거_LQ",
    "직장특화": "직장_LQ",
    "방문특화": "방문_LQ"
}

dong_analysis["상대활동특성"] = (
    dong_analysis[list(lq_cols.values())]
    .idxmax(axis=1)
    .map({v: k for k, v in lq_cols.items()})
)

print("[상대 활동특성]")
print(dong_analysis["상대활동특성"].value_counts())

display(
    dong_analysis[
        [
            "행정동명",
            "주거인구_구성비",
            "직장인구_구성비",
            "방문인구_구성비",
            "주거_LQ",
            "직장_LQ",
            "방문_LQ",
            "상대활동특성"
        ]
    ]
    .sort_values("직장_LQ", ascending=False)
    .head(15)
)

In [ ]:
# 상대 활동특성별 유휴재산 구조
lq_property_summary = (
    dong_analysis
    .groupby("상대활동특성")
    .agg(
        행정동수=("행정동코드", "size"),
        유휴재산수_중앙값=("유휴재산수", "median"),
        총면적_중앙값=("총면적", "median"),
        필지면적_중앙값=("면적중앙값", "median"),
        대_필지수_중앙값=("대_필지수", "median"),
        도로_필지수_중앙값=("도로_필지수", "median")
    )
)

display(lq_property_summary)

In [ ]:
# 부산 전체 연령 구성비 기준
city_under20_share = (
    profile_all["20대미만_활동인구"].sum()
    / profile_all["연령합계_전체활동인구"].sum()
)

city_over60_share = (
    profile_all["60대이상_활동인구"].sum()
    / profile_all["연령합계_전체활동인구"].sum()
)

dong_analysis["20대미만_LQ"] = (
    dong_analysis["20대미만_구성비"] / city_under20_share
)

dong_analysis["60대이상_LQ"] = (
    dong_analysis["60대이상_구성비"] / city_over60_share
)

print("[부산 전체 연령 기준]")
print("20대미만:", round(city_under20_share, 4))
print("60대이상:", round(city_over60_share, 4))

print("\n[20대미만 특화 상위]")
display(
    dong_analysis[
        [
            "행정동명",
            "20대미만_활동인구",
            "20대미만_구성비",
            "20대미만_LQ",
            "유휴재산수",
            "면적중앙값"
        ]
    ]
    .sort_values("20대미만_LQ", ascending=False)
    .head(15)
)

print("\n[60대이상 특화 상위]")
display(
    dong_analysis[
        [
            "행정동명",
            "60대이상_활동인구",
            "60대이상_구성비",
            "60대이상_LQ",
            "유휴재산수",
            "면적중앙값"
        ]
    ]
    .sort_values("60대이상_LQ", ascending=False)
    .head(15)
)

In [ ]:
# 전체 유휴재산의 면적 상위 25% 기준 확인
area_q75 = idle.loc[
    idle["area_valid"],
    "zoneArea"
].quantile(0.75)

print("면적 상위 25% 기준:", area_q75, "㎡")

# 지목과 면적을 이용한 재산조건 보조변수
idle["비도로_확인필지"] = (
    idle["land_type_clean"].notna()
    & (idle["land_type_clean"] != "도로")
)

idle["상위25pct_면적필지"] = (
    idle["area_valid"]
    & (idle["zoneArea"] >= area_q75)
)

idle["비도로_상위25pct"] = (
    idle["비도로_확인필지"]
    & idle["상위25pct_면적필지"]
)

idle["대_상위25pct"] = (
    (idle["land_type_clean"] == "대")
    & idle["상위25pct_면적필지"]
)

print("\n[조건별 필지 수]")
print("비도로 확인필지:", idle["비도로_확인필지"].sum())
print("면적 상위25%:", idle["상위25pct_면적필지"].sum())
print("비도로 + 상위25%:", idle["비도로_상위25pct"].sum())
print("대 + 상위25%:", idle["대_상위25pct"].sum())

In [ ]:
# 행정동별 재산조건 요약
property_capacity = (
    idle.dropna(subset=["행정동코드"])
    .groupby("행정동코드")
    .agg(
        비도로_확인필지수=("비도로_확인필지", "sum"),
        상위25pct_면적필지수=("상위25pct_면적필지", "sum"),
        비도로_상위25pct_필지수=("비도로_상위25pct", "sum"),
        대_상위25pct_필지수=("대_상위25pct", "sum")
    )
    .reset_index()
)

dong_analysis = dong_analysis.merge(
    property_capacity,
    on="행정동코드",
    how="left",
    validate="one_to_one"
)

display(
    dong_analysis[
        [
            "행정동명",
            "유휴재산수",
            "면적중앙값",
            "비도로_확인필지수",
            "상위25pct_면적필지수",
            "비도로_상위25pct_필지수",
            "대_상위25pct_필지수"
        ]
    ]
    .sort_values("비도로_상위25pct_필지수", ascending=False)
    .head(20)
)

In [ ]:
# 코드 형식 통일
profile_all["행정동코드"] = (
    profile_all["행정동코드"]
    .astype("Int64")
    .astype("string")
)

dong_analysis["행정동코드"] = (
    dong_analysis["행정동코드"]
    .astype("Int64")
    .astype("string")
)

# 206개 전체 행정동 기준 LQ
profile_all["20대미만_LQ"] = (
    profile_all["20대미만_구성비"] / city_under20_share
)

profile_all["60대이상_LQ"] = (
    profile_all["60대이상_구성비"] / city_over60_share
)

profile_all["주거_LQ"] = (
    profile_all["주거인구_구성비"] / city_residential_share
)

profile_all["직장_LQ"] = (
    profile_all["직장인구_구성비"] / city_workplace_share
)

profile_all["방문_LQ"] = (
    profile_all["방문인구_구성비"] / city_visit_share
)

# 직장 또는 방문활동이 특화된 정도
profile_all["활동교류_LQ"] = profile_all[
    ["직장_LQ", "방문_LQ"]
].max(axis=1)

# 206개 동 안에서 상대순위
profile_all["아동_규모_pct"] = (
    profile_all["20대미만_활동인구"].rank(pct=True)
)

profile_all["아동_구조_pct"] = (
    profile_all["20대미만_LQ"].rank(pct=True)
)

profile_all["고령_규모_pct"] = (
    profile_all["60대이상_활동인구"].rank(pct=True)
)

profile_all["고령_구조_pct"] = (
    profile_all["60대이상_LQ"].rank(pct=True)
)

profile_all["활동규모_pct"] = (
    profile_all["전체활동인구"].rank(pct=True)
)

profile_all["활동교류_특화_pct"] = (
    profile_all["활동교류_LQ"].rank(pct=True)
)

# 두 조건 중 더 약한 값을 사용
profile_all["아동_수요균형도"] = profile_all[
    ["아동_규모_pct", "아동_구조_pct"]
].min(axis=1)

profile_all["고령_수요균형도"] = profile_all[
    ["고령_규모_pct", "고령_구조_pct"]
].min(axis=1)

profile_all["활동교류_수요균형도"] = profile_all[
    ["활동규모_pct", "활동교류_특화_pct"]
].min(axis=1)

In [ ]:
demand_score_cols = [
    "행정동코드",
    "20대미만_LQ",
    "60대이상_LQ",
    "활동교류_LQ",
    "아동_규모_pct",
    "아동_구조_pct",
    "아동_수요균형도",
    "고령_규모_pct",
    "고령_구조_pct",
    "고령_수요균형도",
    "활동규모_pct",
    "활동교류_특화_pct",
    "활동교류_수요균형도"
]

# 기존 LQ 중복 방지
drop_if_exists = [
    "20대미만_LQ",
    "60대이상_LQ"
]

dong_analysis = dong_analysis.drop(
    columns=[
        c for c in demand_score_cols[1:]
        if c in dong_analysis.columns
    ],
    errors="ignore"
)

dong_analysis = dong_analysis.merge(
    profile_all[demand_score_cols],
    on="행정동코드",
    how="left",
    validate="one_to_one"
)

print("행정동:", len(dong_analysis))
print(
    "수요균형도 결측:",
    dong_analysis[
        [
            "아동_수요균형도",
            "고령_수요균형도",
            "활동교류_수요균형도"
        ]
    ].isna().sum()
)

In [ ]:
common_cols = [
    "행정동명",
    "유휴재산수",
    "면적중앙값",
    "비도로_상위25pct_필지수",
    "대_상위25pct_필지수"
]

print("[아동·가족 생활지원형 후보 지역]")
display(
    dong_analysis[
        common_cols
        + [
            "20대미만_활동인구",
            "20대미만_LQ",
            "아동돌봄시설_전체",
            "아동_수요균형도"
        ]
    ]
    .sort_values("아동_수요균형도", ascending=False)
    .head(20)
)

print("\n[고령 생활지원형 후보 지역]")
display(
    dong_analysis[
        common_cols
        + [
            "60대이상_활동인구",
            "60대이상_LQ",
            "고령층생활서비스_전체",
            "고령_수요균형도"
        ]
    ]
    .sort_values("고령_수요균형도", ascending=False)
    .head(20)
)

print("\n[생활문화·교류형 후보 지역]")
display(
    dong_analysis[
        common_cols
        + [
            "전체활동인구",
            "직장_LQ",
            "방문_LQ",
            "활동교류_LQ",
            "활동교류_수요균형도"
        ]
    ]
    .sort_values("활동교류_수요균형도", ascending=False)
    .head(20)
)

In [ ]:
# 수요와 재산자원의 Pareto frontier 탐색
def get_pareto_mask(df, demand_col, property_col):
    values = df[[demand_col, property_col]].to_numpy(dtype=float)
    mask = np.ones(len(values), dtype=bool)

    for i in range(len(values)):
        dominated = (
            (values[:, 0] >= values[i, 0])
            & (values[:, 1] >= values[i, 1])
            & (
                (values[:, 0] > values[i, 0])
                | (values[:, 1] > values[i, 1])
            )
        ).any()

        mask[i] = not dominated

    return mask


type_settings = {
    "아동·가족 생활지원형": "아동_수요균형도",
    "고령 생활지원형": "고령_수요균형도",
    "활동·교류 지원형": "활동교류_수요균형도"
}

pareto_results = []

for type_name, demand_col in type_settings.items():

    temp = dong_analysis.copy()

    temp["Pareto"] = get_pareto_mask(
        temp,
        demand_col=demand_col,
        property_col="비도로_상위25pct_필지수"
    )

    selected = temp[temp["Pareto"]].copy()
    selected["활용유형"] = type_name
    selected["수요지표"] = selected[demand_col]

    pareto_results.append(selected)

    print(f"\n[{type_name}]")
    print("Pareto 후보 행정동:", len(selected))

    display(
        selected[
            [
                "행정동명",
                demand_col,
                "유휴재산수",
                "비도로_상위25pct_필지수",
                "대_상위25pct_필지수",
                "면적중앙값"
            ]
        ]
        .sort_values(
            [demand_col, "비도로_상위25pct_필지수"],
            ascending=False
        )
    )


pareto_dong = pd.concat(
    pareto_results,
    ignore_index=True
)

In [ ]:
# '대 + 상위25%' 재산만 사용할 경우 Pareto 결과 비교
for type_name, demand_col in type_settings.items():

    temp = dong_analysis.copy()

    temp["Pareto_대"] = get_pareto_mask(
        temp,
        demand_col=demand_col,
        property_col="대_상위25pct_필지수"
    )

    selected = temp[temp["Pareto_대"]]

    print(f"\n[{type_name} - 대 지목 기준]")
    print("후보 행정동:", len(selected))

    display(
        selected[
            [
                "행정동명",
                demand_col,
                "대_상위25pct_필지수",
                "비도로_상위25pct_필지수"
            ]
        ]
        .sort_values(
            [demand_col, "대_상위25pct_필지수"],
            ascending=False
        )
    )

In [ ]:
# 행정동코드 형식을 모든 데이터에서 동일하게 통일
def normalize_dong_code(series):
    return (
        pd.to_numeric(series, errors="coerce")
        .astype("Int64")
        .astype("string")
    )

idle["행정동코드"] = normalize_dong_code(idle["행정동코드"])
dong_analysis["행정동코드"] = normalize_dong_code(dong_analysis["행정동코드"])
pareto_dong["행정동코드"] = normalize_dong_code(pareto_dong["행정동코드"])

print("idle 코드 예시:", idle["행정동코드"].dropna().head().tolist())
print("pareto 코드 예시:", pareto_dong["행정동코드"].dropna().head().tolist())

print(
    "Pareto 코드 중 idle과 매칭:",
    pareto_dong["행정동코드"].isin(idle["행정동코드"]).sum(),
    "/",
    len(pareto_dong)
)

In [ ]:
# 활용유형별 Pareto 지역 내 개별 유휴재산 1차 후보
candidate_list = []

for type_name, demand_col in type_settings.items():

    target_codes = (
        pareto_dong.loc[
            pareto_dong["활용유형"] == type_name,
            "행정동코드"
        ]
        .dropna()
        .unique()
    )

    temp = idle[
        idle["행정동코드"].isin(target_codes)
    ].copy()

    # 원자료는 유지하고, 1차 공간활용 검토군에서만
    # 면적 0·지목 미제공·도로를 제외
    temp = temp[
        temp["area_valid"]
        & temp["land_type_clean"].notna()
        & (temp["land_type_clean"] != "도로")
    ].copy()

    demand_map = (
        dong_analysis
        .set_index("행정동코드")[demand_col]
    )

    temp["수요균형도"] = temp["행정동코드"].map(demand_map)
    temp["활용유형"] = type_name
    temp["상대적대형필지"] = temp["zoneArea"] >= 75
    temp["대지목여부"] = temp["land_type_clean"] == "대"

    candidate_list.append(temp)

property_candidates = pd.concat(
    candidate_list,
    ignore_index=True
)

print("[활용유형별 후보 수]")
print(property_candidates["활용유형"].value_counts())

print("\n전체 후보 행 수:", len(property_candidates))

display(
    property_candidates[
        [
            "활용유형",
            "gugun",
            "행정동명",
            "lctnAddr",
            "zoneArea",
            "land_type_clean",
            "수요균형도",
            "상대적대형필지",
            "대지목여부"
        ]
    ]
    .sort_values(
        ["활용유형", "수요균형도", "zoneArea"],
        ascending=[True, False, False]
    )
    .head(50)
)

In [ ]:
# 같은 재산이 여러 활용유형에 중복 후보로 포함되는지 확인
print("후보 행 수:", len(property_candidates))
print("고유 유휴재산 수:", property_candidates["lctnAddr"].nunique())

type_count = (
    property_candidates
    .groupby("lctnAddr")["활용유형"]
    .nunique()
)

print("\n[활용유형 중복 수]")
print(type_count.value_counts().sort_index())

In [ ]:
# 재산별 활용유형 수요균형도 정리
property_type_scores = (
    property_candidates
    .pivot_table(
        index="lctnAddr",
        columns="활용유형",
        values="수요균형도",
        aggfunc="max"
    )
    .reset_index()
)

property_type_scores.columns.name = None

display(property_type_scores.head())

In [ ]:
type_cols = [
    "아동·가족 생활지원형",
    "고령 생활지원형",
    "활동·교류 지원형"
]

# 후보 유형 개수
property_type_scores["추천가능유형수"] = (
    property_type_scores[type_cols]
    .notna()
    .sum(axis=1)
)

# 가장 높은 수요균형도
property_type_scores["최고_수요균형도"] = (
    property_type_scores[type_cols]
    .max(axis=1)
)

# 대표 추천유형
property_type_scores["대표추천유형"] = (
    property_type_scores[type_cols]
    .idxmax(axis=1)
)

# 추천 가능한 모든 유형 보존
def get_candidate_types(row):
    return " / ".join(
        col for col in type_cols
        if pd.notna(row[col])
    )

property_type_scores["추천가능유형"] = (
    property_type_scores.apply(
        get_candidate_types,
        axis=1
    )
)

print("[대표 추천유형]")
print(
    property_type_scores["대표추천유형"]
    .value_counts()
)

print("\n[추천가능 유형 개수]")
print(
    property_type_scores["추천가능유형수"]
    .value_counts()
    .sort_index()
)

In [ ]:
# 개별 재산 기본정보
property_base = (
    property_candidates[
        [
            "lctnAddr",
            "gugun",
            "행정동명",
            "zoneArea",
            "land_type_clean",
            "x",
            "y",
            "geocode_status"
        ]
    ]
    .drop_duplicates(subset=["lctnAddr"])
)

property_final = property_base.merge(
    property_type_scores,
    on="lctnAddr",
    how="left",
    validate="one_to_one"
)

def classify_review_group(row):

    if (
        row["land_type_clean"] == "대"
        and row["zoneArea"] >= 75
    ):
        return "A_대·상대적대형"

    elif row["zoneArea"] >= 75:
        return "B_비도로·상대적대형"

    else:
        return "C_비도로·소규모"

property_final["재산검토그룹"] = (
    property_final.apply(
        classify_review_group,
        axis=1
    )
)

print("[재산 검토그룹]")
print(
    property_final["재산검토그룹"]
    .value_counts()
)

print("\n[대표 추천유형 × 재산검토그룹]")
display(
    pd.crosstab(
        property_final["대표추천유형"],
        property_final["재산검토그룹"]
    )
)

In [ ]:
# 현재 데이터 기준으로 먼저 살펴볼 A·B 그룹
review_candidates = property_final[
    property_final["재산검토그룹"].isin(
        [
            "A_대·상대적대형",
            "B_비도로·상대적대형"
        ]
    )
].copy()

review_candidates = review_candidates.sort_values(
    [
        "최고_수요균형도",
        "zoneArea"
    ],
    ascending=[False, False]
)

print("A+B 검토 후보:", len(review_candidates))

display(
    review_candidates[
        [
            "대표추천유형",
            "추천가능유형",
            "추천가능유형수",
            "gugun",
            "행정동명",
            "lctnAddr",
            "zoneArea",
            "land_type_clean",
            "최고_수요균형도",
            "재산검토그룹"
        ]
    ]
    .head(50)
)

In [ ]:
pareto_dong.to_csv(
    BASE_PATH + "03_analysis_output/활용유형별_Pareto_후보행정동.csv",
    index=False,
    encoding="utf-8-sig"
)

property_candidates.to_csv(
    BASE_PATH + "03_analysis_output/활용유형별_유휴재산_1차후보.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")

In [ ]:
property_final.to_csv(
    BASE_PATH
    + "03_analysis_output/유휴공유재산_개별활용유형추천_1차.csv",
    index=False,
    encoding="utf-8-sig"
)

review_candidates.to_csv(
    BASE_PATH
    + "03_analysis_output/유휴공유재산_AB검토후보.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")

In [ ]:
# 전체 유휴재산 기준 면적 percentile
valid_areas = (
    idle.loc[idle["area_valid"], "zoneArea"]
    .sort_values()
    .to_numpy()
)

def area_percentile(value):
    return np.searchsorted(
        valid_areas,
        value,
        side="right"
    ) / len(valid_areas)

property_final["면적_pct"] = (
    property_final["zoneArea"]
    .apply(area_percentile)
)

print(
    property_final[
        ["zoneArea", "면적_pct"]
    ].describe()
)

In [ ]:
primary_candidates = property_final[
    property_final["재산검토그룹"] == "A_대·상대적대형"
].copy()

print("1차 우선 검토군:", len(primary_candidates))

print("\n[대표추천유형]")
print(
    primary_candidates["대표추천유형"]
    .value_counts()
)

In [ ]:
type_score_map = {
    "아동·가족 생활지원형": "아동·가족 생활지원형",
    "고령 생활지원형": "고령 생활지원형",
    "활동·교류 지원형": "활동·교류 지원형"
}

ranking_rows = []

for type_name, score_col in type_score_map.items():

    temp = primary_candidates[
        primary_candidates[score_col].notna()
    ].copy()

    temp["유형별_수요균형도"] = temp[score_col]

    # 수요와 재산규모 중 더 약한 조건 사용
    temp["우선검토균형도"] = temp[
        [
            "유형별_수요균형도",
            "면적_pct"
        ]
    ].min(axis=1)

    temp["검토활용유형"] = type_name

    ranking_rows.append(temp)

priority_by_type = pd.concat(
    ranking_rows,
    ignore_index=True
)

priority_by_type["유형내순위"] = (
    priority_by_type
    .groupby("검토활용유형")["우선검토균형도"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)

print("[유형별 A군 후보]")
print(
    priority_by_type["검토활용유형"]
    .value_counts()
)

In [ ]:
for type_name in type_score_map:

    print(f"\n[{type_name}]")

    display(
        priority_by_type[
            priority_by_type["검토활용유형"] == type_name
        ][
            [
                "유형내순위",
                "gugun",
                "행정동명",
                "lctnAddr",
                "zoneArea",
                "land_type_clean",
                "유형별_수요균형도",
                "면적_pct",
                "우선검토균형도",
                "추천가능유형"
            ]
        ]
        .sort_values(
            [
                "우선검토균형도",
                "유형별_수요균형도",
                "zoneArea"
            ],
            ascending=False
        )
        .head(15)
    )

In [ ]:
multi_priority = (
    priority_by_type
    .groupby("lctnAddr")
    .agg(
        우선검토유형수=("검토활용유형", "nunique"),
        최고_우선검토균형도=("우선검토균형도", "max")
    )
    .reset_index()
)

print("[A군의 활용유형 중복]")
print(
    multi_priority["우선검토유형수"]
    .value_counts()
    .sort_index()
)

In [ ]:
priority_by_type.to_csv(
    BASE_PATH
    + "03_analysis_output/활용유형별_A군_우선검토순위.csv",
    index=False,
    encoding="utf-8-sig"
)

primary_candidates.to_csv(
    BASE_PATH
    + "03_analysis_output/유휴공유재산_A군_186건.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")

In [ ]:
# 활용유형 × 행정동별 대표 유휴재산 선정
# 동일 행정동에서는 우선검토균형도 → 수요 → 면적 순으로 대표 1건 선정

representative_candidates = (
    priority_by_type
    .sort_values(
        [
            "검토활용유형",
            "우선검토균형도",
            "유형별_수요균형도",
            "면적_pct",
            "zoneArea"
        ],
        ascending=[True, False, False, False, False]
    )
    .drop_duplicates(
        subset=["검토활용유형", "행정동명"],
        keep="first"
    )
    .copy()
)

# 최종 표에서 사용할 순번
representative_candidates = (
    representative_candidates
    .sort_values(
        [
            "검토활용유형",
            "우선검토균형도",
            "유형별_수요균형도",
            "zoneArea"
        ],
        ascending=[True, False, False, False]
    )
)

representative_candidates["대표후보순번"] = (
    representative_candidates
    .groupby("검토활용유형")
    .cumcount()
    + 1
)

# 유형별 상위 5개
final_shortlist = representative_candidates[
    representative_candidates["대표후보순번"] <= 5
].copy()

for type_name in [
    "아동·가족 생활지원형",
    "고령 생활지원형",
    "활동·교류 지원형"
]:
    print(f"\n[{type_name}]")

    display(
        final_shortlist[
            final_shortlist["검토활용유형"] == type_name
        ][
            [
                "대표후보순번",
                "gugun",
                "행정동명",
                "lctnAddr",
                "zoneArea",
                "land_type_clean",
                "유형별_수요균형도",
                "면적_pct",
                "우선검토균형도",
                "추천가능유형"
            ]
        ]
    )

# 저장
representative_candidates.to_csv(
    BASE_PATH
    + "03_analysis_output/활용유형별_행정동대표후보.csv",
    index=False,
    encoding="utf-8-sig"
)

final_shortlist.to_csv(
    BASE_PATH
    + "03_analysis_output/활용유형별_대표후보_TOP5.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n대표 후보 저장 완료")

# ✅ 유휴 공유재산 중심 EDA 결론

## 1. 유휴재산의 물리적 조건은 매우 이질적임

전체 10,102건의 유휴 공유재산은
면적과 지목에서 큰 차이를 보였다.

- 면적 중앙값: 27㎡
- 전체 재산의 상위 25% 면적 기준: 75㎡
- 대 지목: 4,871건
- 도로 지목: 3,296건

따라서 모든 재산에 동일한 활용기준을 적용하기보다
재산조건과 지역수요를 함께 고려할 필요가 있다.

---

## 2. 지역수요와 유휴재산 보유량은 단순하게 대응하지 않음

행정동 단위 분석 결과,
생활수요와 유휴재산 수·면적 사이에
강한 단순 상관관계는 나타나지 않았다.

또한 높은 생활수요를 가진 지역이라고 하더라도
실제 보유한 유휴재산의 면적과 지목조건은 서로 달랐다.

이는

> 수요가 높은 지역 = 활용 가능한 유휴재산이 많은 지역

으로 볼 수 없음을 의미한다.

---

## 3. 세 가지 활용수요 축을 설정

EDA 결과를 바탕으로 다음 3개 활용수요 축을 설정하였다.

1. 아동·가족 생활지원형
2. 고령 생활지원형
3. 활동·교류 지원형

지역수요는 절대 규모와 지역 내 상대적 특성을 함께 반영하고,
재산조건과 별도로 평가하였다.

---

## 4. 지역수요와 재산조건의 교집합을 탐색

지역수요와 상대적으로 큰 비도로 유휴재산 보유량을
Pareto 방식으로 비교하여 후보 행정동을 탐색하였다.

이후 개별 재산 수준에서

- 지목
- 면적
- 지역수요
- 활용유형

을 결합하였다.

최종적으로 1,572개의 고유 유휴재산이
하나 이상의 활용유형 후보로 도출되었다.

이 가운데

- 대 지목
- 75㎡ 이상

조건을 동시에 만족하는 186건을
현재 확보된 데이터 기준의 **1차 우선 검토군(A군)** 으로 구분하였다.

※ 75㎡는 법적 또는 시설 설치 최소면적이 아니라
전체 유휴재산 분포의 상위 25%를 구분하기 위한 상대기준이다.

---

## 5. 활용유형별 대표 후보

### 아동·가족 생활지원형
- 부산진구 부암3동 / 부암동 507-2 / 991㎡ / 대
- 부산진구 가야1동 / 가야동 82-1 / 1,200㎡ / 대

### 고령 생활지원형
- 해운대구 반송2동 / 반송동 219-13 / 131㎡ / 대
- 부산진구 초읍동 / 초읍동 291-38 / 727.3㎡ / 대

### 활동·교류 지원형
- 부산진구 부전1동 / 부전동 25-1 / 616.5㎡ / 대
- 부산진구 전포1동 / 전포동 343-5 / 1,221.8㎡ / 대

---

## 핵심 해석

본 분석에서 중요한 것은
수요가 높은 지역이나 면적이 큰 재산을 각각 찾는 것이 아니다.

> **지역의 필요 × 재산의 조건**

이 동시에 충족되는 유휴 공유재산을 찾는 것이 핵심이다.

따라서 최종 결과는 활용 가능성을 확정하는 것이 아니라,
향후 법적·물리적·사업성 검토가 우선적으로 필요한
**우선 검토 대상**으로 해석한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd

path = '/content/drive/MyDrive/부산_BigData_공모전/02_processed_data/유휴공유재산_16개구군_통합.csv'
df = pd.read_csv(path)

# 전체 면적자료
all_area = df['zoneArea']

# 0㎡ 제외 유효 면적자료
valid_area = df.loc[df['zoneArea'] > 0, 'zoneArea']

print('[전체 면적자료]')
print('건수:', len(all_area))
print('중앙값:', all_area.median())
print('75백분위:', all_area.quantile(0.75))

print('\n[0㎡ 제외 유효 면적자료]')
print('건수:', len(valid_area))
print('중앙값:', valid_area.median())
print('75백분위:', valid_area.quantile(0.75))

print('\n[0㎡ 이하 확인]')
print(df.loc[df['zoneArea'] <= 0, ['gugun', 'lctnAddr', 'landLdcgNm', 'zoneArea']])